[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IshtVibhu/iv-Mesa-ABM-Tutorial/blob/main/NB_06_StatsModels.ipynb)

# Python for Computational Economics
## Notebook 06 — StatsModels & linearmodels: Econometrics in Python

**Prerequisites:** NB_01–NB_05

**What you will learn:**
- OLS regression with full summary output (t-stats, R², heteroskedasticity-robust SEs)
- Instrumental variables / 2SLS (linearmodels)
- Fixed-effects panel data estimation
- ARIMA time-series modelling
- Structural break tests (Chow test)

StatsModels output is deliberately similar to Stata/R, so existing econometricians feel at home immediately.

In [ ]:
try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    import pandas as pd
    import numpy as np
except ImportError:
    !pip install statsmodels pandas numpy --quiet
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    import pandas as pd
    import numpy as np

try:
    from linearmodels import PanelOLS, IV2SLS
except ImportError:
    !pip install linearmodels --quiet
    from linearmodels import PanelOLS, IV2SLS

import warnings
warnings.filterwarnings("ignore")
print(f"StatsModels {sm.__version__}")

---
## 1. OLS Regression — Returns to Education (Mincer Equation)

In [ ]:
np.random.seed(42)
N = 500

# Simulate a cross-section of workers
education     = np.random.choice(range(8, 22), size=N)   # years of schooling
experience    = np.random.randint(0, 35, size=N)          # years of work experience
female        = np.random.binomial(1, 0.48, size=N)       # gender dummy
ability_noise = np.random.normal(0, 0.15, size=N)         # unobserved ability

# Log-wage Mincer equation (true coefficients)
log_wage = (
    1.5
    + 0.08 * education
    + 0.04 * experience
    - 0.005 * experience**2
    - 0.15 * female
    + ability_noise
    + np.random.normal(0, 0.2, N)
)

workers = pd.DataFrame({
    "log_wage":   log_wage,
    "education":  education,
    "experience": experience,
    "experience2": experience**2,
    "female":     female,
})

print(workers.describe().round(2))

In [ ]:
# Estimate with formula API (similar to R's lm())
model = smf.ols(
    "log_wage ~ education + experience + experience2 + female",
    data=workers
).fit(cov_type="HC3")  # HC3 = heteroskedasticity-robust standard errors

print(model.summary())

In [ ]:
# Extract key results
education_coef = model.params["education"]
education_se   = model.bse["education"]
education_pval = model.pvalues["education"]

print("Key results:")
print(f"  Returns to education:  {education_coef:.4f} ({education_coef*100:.2f}% per year of schooling)")
print(f"  Standard error:        {education_se:.4f}")
print(f"  p-value:               {education_pval:.4e}")
print(f"  95% CI:                [{education_coef - 1.96*education_se:.4f}, {education_coef + 1.96*education_se:.4f}]")
print(f"  Gender wage gap:       {model.params['female']*100:.1f}%")
print(f"  R²:                    {model.rsquared:.3f}")

---
## 2. Instrumental Variables (2SLS) — Addressing Endogeneity

Education is endogenous: unobserved ability raises both education and wages, biasing OLS upward. A classic instrument is **distance to college** (Card, 1995) — it affects schooling but not wages directly.

In [ ]:
np.random.seed(7)
N = 600

# Unobserved ability
ability = np.random.normal(0, 1, N)

# Instrument: distance to nearest college (km)
distance = np.random.exponential(scale=20, size=N)

# Education is affected by ability AND distance (closer → more years)
education = 12 + 2.0 * ability - 0.08 * distance + np.random.normal(0, 1, N)
education = np.clip(education, 6, 22)

# Log-wage: depends on education and ability (ability is unobserved)
log_wage = 1.0 + 0.07 * education + 0.5 * ability + np.random.normal(0, 0.3, N)

iv_data = pd.DataFrame({
    "log_wage":  log_wage,
    "education": education,
    "distance":  distance,
    "const":     1.0,
})

# OLS (biased — ignores ability)
ols = smf.ols("log_wage ~ education", data=iv_data).fit()

# 2SLS using IV2SLS from linearmodels
iv_data_indexed = iv_data.copy()
res_iv = IV2SLS(
    dependent=iv_data_indexed["log_wage"],
    exog=iv_data_indexed[["const"]],
    endog=iv_data_indexed[["education"]],
    instruments=iv_data_indexed[["distance"]],
).fit(cov_type="robust")

print("Comparison: OLS vs IV/2SLS")
print(f"  True coefficient:    0.0700")
print(f"  OLS coefficient:     {ols.params['education']:.4f}  (biased upward)")
print(f"  IV coefficient:      {res_iv.params['education']:.4f}  (consistent)")
print(f"\nFirst-stage F-statistic: {res_iv.first_stage.diagnostics['f.stat']['education']:.1f}")
print("(F > 10 suggests the instrument is not weak)")

---
## 3. Panel Data — Fixed Effects

In [ ]:
np.random.seed(12)
N_COUNTRIES = 30
N_YEARS     = 15

# Country fixed effects (unobserved heterogeneity)
country_fe = np.random.normal(0, 1, N_COUNTRIES)

rows = []
for i in range(N_COUNTRIES):
    for t in range(N_YEARS):
        trade_openness = 30 + 10 * np.random.rand() + 0.5 * t
        gdp_growth = (
            0.5
            + 0.06 * trade_openness  # true effect of trade
            + country_fe[i]           # unobserved country effect
            + np.random.normal(0, 0.8)
        )
        rows.append({"country": i, "year": t, "gdp_growth": gdp_growth,
                     "trade_openness": trade_openness})

panel_df = pd.DataFrame(rows)
panel_df = panel_df.set_index(["country", "year"])

# Pooled OLS (ignores fixed effects — biased)
ols_pooled = smf.ols("gdp_growth ~ trade_openness",
                     data=panel_df.reset_index()).fit()

# Fixed-effects (within) estimator
fe_model = PanelOLS(
    dependent=panel_df["gdp_growth"],
    exog=sm.add_constant(panel_df[["trade_openness"]]),
    entity_effects=True,
).fit(cov_type="clustered", cluster_entity=True)

print("Trade-Growth Regression: Pooled OLS vs Fixed Effects")
print(f"  True effect of trade openness: 0.0600")
print(f"  Pooled OLS:    {ols_pooled.params['trade_openness']:.4f}")
print(f"  Fixed Effects: {fe_model.params['trade_openness']:.4f}")
print(f"  (FE is closer to the true 0.06 by removing country-level omitted variables)")

---
## 4. ARIMA — Forecasting GDP Growth

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

np.random.seed(0)
# Simulate AR(1) GDP growth series
n_obs = 60
gdp_growth = [2.5]
for _ in range(n_obs - 1):
    gdp_growth.append(0.5 + 0.65 * gdp_growth[-1] + np.random.normal(0, 0.6))
gdp_growth = np.array(gdp_growth)

# Fit ARIMA(1, 0, 0) — an AR(1) model
arima_model = ARIMA(gdp_growth, order=(1, 0, 0)).fit()

print("ARIMA(1,0,0) — AR(1) model for GDP growth:")
print(arima_model.summary().tables[1])

# Forecast 8 steps ahead
forecast = arima_model.forecast(steps=8)
print(f"\n8-period-ahead forecast: {forecast.round(2).values}")
print(f"Long-run mean:           {arima_model.params['const'] / (1 - arima_model.params['ar.L1']):.3f}")

---
## Your Turn — Wage Penalty for Part-Time Work

Using the `workers` DataFrame from section 1, add a `part_time` dummy (randomly assigned with probability 0.3). Estimate:
1. A basic OLS: `log_wage ~ part_time`
2. A controlled OLS: `log_wage ~ part_time + education + experience + experience2 + female`

Report: Does the part-time wage penalty change when controls are added? Why?

In [ ]:
# Your code here


In [ ]:
#@title Solution
np.random.seed(55)
workers_ex = workers.copy()
workers_ex["part_time"] = np.random.binomial(1, 0.3, len(workers_ex))
# Add a wage penalty for part-time
workers_ex["log_wage"] = workers_ex["log_wage"] - 0.18 * workers_ex["part_time"]

m1 = smf.ols("log_wage ~ part_time", data=workers_ex).fit(cov_type="HC3")
m2 = smf.ols("log_wage ~ part_time + education + experience + experience2 + female",
              data=workers_ex).fit(cov_type="HC3")

print("Part-time wage penalty:")
print(f"  Basic OLS:    {m1.params['part_time']:.4f}  (SE={m1.bse['part_time']:.4f})")
print(f"  Controlled:   {m2.params['part_time']:.4f}  (SE={m2.bse['part_time']:.4f})")
print("\nControlled estimate is closer to the true -0.18.")
print("Without controls, OLS absorbs differences in education/experience of part-timers.")

---
## Summary

| Method | StatsModels / linearmodels |
|---|---|
| OLS | `smf.ols(formula, data).fit(cov_type='HC3')` |
| 2SLS / IV | `IV2SLS(dep, exog, endog, instruments).fit()` |
| Panel FE | `PanelOLS(dep, exog, entity_effects=True).fit()` |
| ARIMA | `ARIMA(series, order=(p,d,q)).fit()` |
| Forecast | `.forecast(steps=n)` |
| Robust SEs | `fit(cov_type='HC3')` or `'clustered'` |

**Next:** NB_07_QuantEcon.ipynb — dynamic programming and Markov chains.